# Método de disparo

Este método convierte el problema de valores en la frontera en un problema de valor inicial mediante la determinación de los valores iniciales faltantes que son consistentes con los valores de la frontera.

Un problema pide por ejemplo en el intervalo $a \le t \le b$: 

$$\begin{cases} y'' = f(t,y,y') \\ y(a) = y_a \\ y(b) = y_b \end{cases} $$

El método de disparo resuelve el PVF al encontrar el PVI que tiene la misma solución

Esto puede hacerse numéricamente de la siguiente forma, dada la condición de frontera $y(b) = b$. Hacemos una primera suposición $s$ de forma que debe resolverse el PVI:

$$\begin{cases}y'' = f(t,y,y') \\ y(a) = y_a \\ y'(a) = s \end{cases}$$

Entonces el PVI correcto es cuando $F(s) = \lvert y_b - y(b) \rvert = 0$


Ejemplo:

Resuelve el PVF

$$\begin{cases} y'' = 4y \\ y(0) = 1 \\ y(1) = 3\end{cases} $$



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sistema(Y, t):
    Y1, Y2 = Y

    DY2 = 4*Y1
    DY1 = Y2

    return np.array([DY1, DY2])

def rk4(sistema, a, b, h, Y0):

    t = np.arange(a, b + h, h)
    n = t.shape[0]
    Y = np.zeros((n, Y0.shape[0]))
    Y[0,:] = Y0

    for i in range(n-1):
        k1 = sistema(Y[i,:],t[i])
        k2 = sistema(Y[i,:] + h*k1/2, t[i] + h/2)
        k3 = sistema(Y[i,:] + h*k2/2, t[i] + h/2)
        k4 = sistema(Y[i,:] + h*k3, t[i] + h)

        Y[i+1,:] = Y[i,:] + (h/6)*(k1 + 2*k2 + 2*k3 + k4)

    return Y, t

def metodo_secante(f, x0, x1, N, tol):

    print(f"{'n':<{3}}{'x':<{15}}{'f(x)':<{15}}{'error':<{15}}")
    
    for i in range(N):

        y0 = f(x0)
        y1 = f(x1)

        m = (y1 - y0)/(x1 - x0)

        x = x1 - y1/m

        error = abs((x - x1)/max(abs(x), 1e-15))

        print(f"{i+1:<{3}}{x:<{15}.10f}{f(x):<{15}.10f}{error:<{15}e}")

        if error < tol:
            return x, error
        
        x0 = x1
        x1 = x
    
    print('No se pudo aproximar la raiz con la precisión solicitada')
    return x, error

a = 0
b = 1
y0 = 1
yb = 3
s0 = 0
h = 0.01

def generador(a, b, y0, yb, sistema, integrador, h):

    def f(s):

        Y0 = np.array([y0, s])

        Y, _ = integrador(sistema, a, b, h ,Y0)

        return yb - Y[-1, 0]
    
    return f

f = generador(a, b, y0, yb, sistema, rk4, h)

s, error = metodo_secante(f, 0, 1, 100, 1e-6)

def exacta(t):
    return (3 - np.exp(-2))/(np.exp(2) - np.exp(-2))*np.exp(2*t) + (np.exp(2) - 3)/(np.exp(2) - np.exp(-2))*np.exp(-2*t)

Y0 = np.array([y0, s])
Y, t = rk4(sistema, a, b, h, Y0)

Y_e = exacta(t)

fig, ax = plt.subplots(figsize = (6,4))

ax.plot(t, Y_e, c = 'blue', lw = 2, label = 'Exacta')
ax.plot(t, Y[:,0], c = 'red', lw = 2, ls = '--', label = 'Aproximada')
ax.legend()

ax.grid(alpha = 0.3)

plt.tight_layout()
plt.show()





In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sistema(Y, t):
    Y1, Y2 = Y
    DY1 = (4-2*Y2)/t**3
    DY2 = -np.exp(Y1)

    return np.array([DY1, DY2])

def rk4(sistema, a, b, h, Y0):
    t = np.arange(a, b + h, h)
    n = t.shape[0]
    Y = np.zeros((n,Y0.shape[0]))
    Y[0,:] = Y0

    for i in range(n-1):
        k1 = sistema(Y[i,:], t[i])
        k2 = sistema(Y[i,:] + h*k1/2, t[i] + h/2)
        k3 = sistema(Y[i,:] + h*k2/2, t[i] + h/2)
        k4 = sistema(Y[i,:] + h*k3, t[i] + h)

        Y[i+1,:] = Y[i,:] + (h/6)*(k1 + 2*k2 + 2*k3 + k4)

    return Y, t

def metodo_secante(f, x0, x1, N, tol):

    print(f"{'n':<{3}}{'x':<{15}}{'f(x)':<{15}}{'error':<{15}}")

    y0 = f(x0)
    y1 = f(x1)

    for i in range(N):

        m = (y1 - y0)/(x1 - x0)

        x = x1 - y1/m

        error = abs((x - x1)/max(abs(x), 1e-15))

        y = f(x)

        print(f"{i+1:<{3}}{x:<{15}.10f}{y:<{15}.10f}{error:<{15}e}")

        if error < tol:
            return x, error
        
        x0, y0 = x1, y1
        x1, y1 = x, y
        
    print('No se pudo aproximar la raiz con la precisión solicitada')

    return x, error

a = 1
b = 2
y_a = 0
y_b = 0
h = 0.01

def generador(sistema, integrador, a, b, y_a, y_b, h):

    def f(s):
        Y0 = np.array([y_a, s])
        Y, _ = integrador(sistema, a, b, h, Y0)

        return y_b - Y[-1, 1]
    
    return f
def y1_e(t):
    return np.log(t)

def y2_e(t):
    return 2 - t**2/2

f = generador(sistema, rk4, a, b, y_a, y_b, h)

s, error = metodo_secante(f, 0, 1, 100, 1e-6)

Y0 = np.array([y_a, s])
Y, t = rk4(sistema, a, b, h, Y0)
Y1_e = y1_e(t)
Y2_e = y2_e(t)

fig, ax = plt.subplots(figsize = (6,4))
ax.plot(t, Y1_e, c = 'blue', lw = 2, label = 'Exacta')
ax.plot(t, Y2_e, c = 'orange', lw = 2, label = 'Exacta')
ax.plot(t, Y[:,0], c = 'red', lw = 2, ls = '--', label = 'Aproximada')
ax.plot(t, Y[:,1], c = 'green', lw = 2, ls = '--', label = 'Aproximada')
ax.legend()
ax.grid(alpha = 0.3)
plt.tight_layout()
plt.show()



# Diferencias finitas

La idea principal detrás de los métodos de diferencias finitas es reemplazar las derivadas de la ecuación diferencial por aproximaciones discretas y evaluar sobre una malla para desarrollar un sistema de ecuaciones.

Para ellos haremos uso de las diferencias centradas:

$$y'(t) = \frac{y(t+h) - y(t-h)}{2h} - \frac{h^2}{6}y'''(c)$$

$$y''(t) = \frac{y(t+h) - 2y(t) + y(t-h)}{h^2} + \frac{h^2}{12}f''''(c)$$

Ambas son precisas hasta un error profesional a $h^2$.

Después de las sustituciones hay dos posibles situaciones. Si el problema de valor de frontera original era lineal, entonces el sistema resultante de ecuaciones es lineal y puede resolver por eliminación de Gauss o por medio de métodos iterativos. Si el problema original era no lineal, entonces el sistema alegbraico es un sistema de ecuaciones no lineales que requiere técnicas más sofisticadas.


Resuelva:

$$\begin{cases} y'' = 4y \\ y(0) = 1 \\ y(1) = 3 \end{cases}$$

In [ ]:
import numpy as np

def matriz_coeficientes(h, n):
    W = np.zeros((n,n))
    for i in range(n):
        W[i, i] = -4*h**2 - 2
        if i != 0:
            W[i, i - 1] = 1
        if i != n - 1:
            W[i, i + 1] = 1
    return W

a = 0
b = 1
ya = 1
yb = 3
n = 5
t = np.linspace(a, b, n)
h = (b-a)/(n-1)

A = matriz_coeficientes(h, n-2)

print(A)

b = np.zeros(n-2)
b[0] = - ya
b[-1] = -yb

def metodo_thomas(matriz, vector):
    A = matriz.copy()
    b = vector.copy()
    n = b.shape[0]

    P = np.zeros(n-1)
    Q = np.zeros(n)

    P[0] = A[0, 1]/A[0,0]
    Q[0] = b[0]/A[0,0]

    for i in range(1, n):
        denom = (A[i,i] - A[i,i-1]*P[i-1])
        if i != (n-1):
            P[i] = A[i,i+1]/denom
        Q[i] = (b[i] - A[i,i-1]*Q[i-1])/denom

    x = np.zeros(n)

    x[n-1] = Q[n-1]

    for i in range(n-2, -1, -1):

        x[i] = Q[i] - P[i]*x[i+1]

    return x

y_interior = metodo_thomas(A, b)

y = np.zeros(n)
y[0] = ya
y[-1] = yb
y[1:n-1] = y_interior

print(y)

        


Resuelve:

$$\begin{cases}
y''(x) = -\pi sin(\pi x) \\ y(0) = 0 \\ y(1) = 0 
\end{cases}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sistema(n):

    W = np.zeros((n,n))

    for i in range(n):
        W[i,i] = -2
        if i != 0:
            W[i,i-1] = 1
        if i != (n-1):
            W[i,i+1] = 1
    return W
a = 0
ya = 0
b = 1
yb = 0
n = 100

x = np.linspace(a, b, n)
h = (b - a)/(n - 1)

b = -np.pi**2*h**2*np.sin(np.pi*x[1:n-1])

W = sistema(n - 2)

def metodo_thomas(matriz, vector):
    A = matriz.copy()
    b = vector.copy()
    n = b.shape[0]

    P = np.zeros(n-1)
    Q = np.zeros(n)

    P[0] = A[0, 1]/A[0,0]
    Q[0] = b[0]/A[0,0]

    for i in range(1, n):
        denom = (A[i,i] - A[i,i-1]*P[i-1])
        if i != (n-1):
            P[i] = A[i,i+1]/denom
        Q[i] = (b[i] - A[i,i-1]*Q[i-1])/denom

    x = np.zeros(n)

    x[n-1] = Q[n-1]

    for i in range(n-2, -1, -1):

        x[i] = Q[i] - P[i]*x[i+1]

    return x

y_interior = metodo_thomas(W, b)

y = np.zeros(n)
y[0] = ya
y[-1] = yb
y[1:-1] = y_interior

def exacta(x):
    return np.sin(np.pi*x)

y_e = exacta(x)

fig, ax = plt.subplots(figsize = (8,6))

ax.plot(x, y_e, label = 'Exacta', lw = 2, c = 'red')
ax.plot(x, y, label = 'Aproximación', lw = 2 , c = 'blue', ls = '--')
ax.grid(alpha = 0.3)
ax.legend()
plt.tight_layout
plt.show()












Resuelva:

$$
\begin{cases}

y'' + 2y' - 3y = e^x \\
y(0) = 1 \\
y(1) = 2

\end{cases}
$$

In [ ]:
import numpy as np

def sistema(h, n):
    W = np.zeros((n,n)) 

    for i in range(n):
        W[i,i] = -2 - 3*h**2
        if i != 0:
            W[i,i-1] = 1 - h
        if i != (n-1):
            W[i,i+1] = 1 + h

    return W

a = 0
b = 1
ya = 1
yb = 2
n = 100
x = np.linspace(a, b, n)

h = (b - a)/(n - 1)

b = (h**2*np.exp(x[1:-1]))
b[0] = b[0] - (1-h)
b[-1] = b[-1] - 2*(1+h)

W = sistema(h, n - 2)

def metodo_thomas(matriz, vector):
    A = matriz.copy()
    b = vector.copy()
    n = b.shape[0]

    P = np.zeros(n-1)
    Q = np.zeros(n)

    P[0] = A[0, 1]/A[0,0]
    Q[0] = b[0]/A[0,0]

    for i in range(1, n):
        denom = (A[i,i] - A[i,i-1]*P[i-1])
        if i != (n-1):
            P[i] = A[i,i+1]/denom
        Q[i] = (b[i] - A[i,i-1]*Q[i-1])/denom

    x = np.zeros(n)

    x[n-1] = Q[n-1]

    for i in range(n-2, -1, -1):

        x[i] = Q[i] - P[i]*x[i+1]

    return x

y_interior = metodo_thomas(W, b)

y = np.zeros(n)
y[0] = ya
y[-1] = yb
y[1:-1] = y_interior

def exacta(x):
    return ((2 - 1/4*np.exp(1) - np.exp(-3))/(np.exp(1) - np.exp(-3)))*np.exp(x) + (1 - (2 - 1/4*np.exp(1) - np.exp(-3))/(np.exp(1) - np.exp(-3)))*np.exp(-3*x) + 1/4*x*np.exp(x)

y_e = exacta(x)

fig, ax = plt.subplots(figsize = (8,6))

ax.plot(x, y_e, label = 'Exacta', lw = 2, c = 'red')
ax.plot(x, y, label = 'Aproximación', lw = 2 , c = 'blue', ls = '--')
ax.grid(alpha = 0.3)
ax.legend()
plt.tight_layout
plt.show()



